# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset, described by a Croissant schema, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed (uncomment below if running live)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata = dataset.metadata  # .metadata is a single object, not a dict
print(f"Dataset title: {metadata.name}\nDescription: {metadata.description}")

# Show publication date, citation, and Croissant version if available
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Croissant schema version: {getattr(metadata, 'conformsTo', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

The Croissant schema organizes tabular data into one or more **record sets** (tables/files), each of which exposes one or more **fields** (columns). All entities, including record sets, fields, and columns, are referenced by their unique `@id`.

Let's enumerate all record sets defined by the dataset:

In [ ]:
# List all record sets
record_sets = list(dataset.record_sets.keys())
print(f"Found {len(record_sets)} record set(s):")
for rs_id in record_sets:
    rs = dataset.record_sets[rs_id]
    print(f"- RecordSet @id: {rs_id}\n  Name: {rs.name if hasattr(rs, 'name') else 'N/A'}\n  Description: {getattr(rs, 'description', 'N/A')}")
    if hasattr(rs, 'fields'):
        print("  Fields:")
        for field in rs.fields:
            print(f"    - Field @id: {field.id} | Name: {field.name} | DataType: {getattr(field, 'dataType', 'N/A')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

**Note:** We'll select the main record set for this dataset (based on the schema naming, it's commonly the first listed).

Let's load records for the first record set and inspect the columns.

In [ ]:
# Use all available record set IDs
all_record_set_ids = list(dataset.record_sets.keys())
dataframes = {}

for record_set_id in all_record_set_ids:
    print(f"\nLoading records from RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records. Columns (fields by @id):\n{df.columns.tolist()}")


Let's display the first few rows of the main record set.

In [ ]:
# For demonstration, use the first record set as the "main" table
main_record_set_id = all_record_set_ids[0] if all_record_set_ids else None

if main_record_set_id is not None:
    display_cols = dataframes[main_record_set_id].columns.tolist()
    print(f"@id: {main_record_set_id}\nDisplaying first 5 records:")
    display(dataframes[main_record_set_id].head())
else:
    print('No record sets found in the dataset.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, or grouping data. Operations such as removing outliers, transforming fields, or grouping by key attributes can be applied here to prepare for further analysis.

First, let's look for a numeric field in the main record set (a typical candidate would be protein abundance, peptide counts, or molecular weight).

Below, update the variables with the desired field `@id` for your analysis.

In [ ]:
# --- User selection: Identify a numeric field for demo analysis ---
df = dataframes[main_record_set_id]
numeric_fields = df.select_dtypes(include='number').columns.tolist()
print(f"Numeric fields (by @id): {numeric_fields}")

# For illustration, pick the first numeric field (if available)
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    print("No numeric columns found.")
    numeric_field_id = None

if numeric_field_id is not None:
    # Filter: values greater than a threshold
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} of {len(df)}\n")

    # Z-normalize the field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"First 5 normalized values for {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field if present
    cat_fields = df.select_dtypes(include='object').columns.tolist()
    group_field = cat_fields[0] if cat_fields else None
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field} (first 5):")
        display(grouped_df.head())
else:
    print('No numeric field found for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using common charting tools (e.g., matplotlib, seaborn). Here, we display a histogram of the selected numeric field and a boxplot grouped by a categorical field if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30, color='steelblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we explored the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset using `mlcroissant`. We loaded its metadata, traversed the record sets and fields using their `@id`s, extracted tabular records, and performed basic exploratory data analysis including normalization and visualization.

You can extend the analysis with additional statistical tests, filtering, or domain-specific visualizations relevant to proteomics and mass spectrometry data.